# 06 · `gl_engine/resolve/book.py`

## What this file is for

[`05-resolve-resolver`](05-resolve-resolver.ipynb) decided *which two packages govern*. This file makes that pair usable: given a name, which layer actually supplies it?

The rule is one sentence, and everything else follows from it: **the state layer overrides the countrywide layer by name, wholesale — even when the state's version is empty.** An empty override means *not offered in this state*. It does not mean *look upstream*.

**Depends on:** [`04-erc-discovery`](04-erc-discovery.ipynb), [`05-resolve-resolver`](05-resolve-resolver.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.resolve import book

for name, obj in vars(book).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != book.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Build a book for one state on one date, then ask who owns a table.

In [ ]:
from gl_engine import EditionResolver
from gl_engine.resolve.book import ResolvedBook

r = EditionResolver()
book = ResolvedBook(r.resolve("GA", "20260811"))

print("book      :", book.juris, "@", book.asof)
print("state     :", book.state.pkg_id)
print("countrywide:", book.parent.pkg_id)
print()
name = "DedFactorProdsCSL"
print(f"{name} is owned by:", book.declares(name, "Rate"))

`declares()` answers *which layer supplies this*, and it is the question you want when a number surprises you. Georgia files no deductible factors of its own — every one of them comes from the countrywide layer.

## The interesting case

### How much does a state actually override?

Most of ISO's content is national. A state package is an exception list, and it is usually a short one.

In [ ]:
from gl_engine.erc.tables import list_tables

state_rate  = set(list_tables(book.state.package.content, "Rate"))
parent_rate = set(list_tables(book.parent.package.content, "Rate"))

print(f"rate tables in the state package      : {len(state_rate)}")
print(f"rate tables in the countrywide package: {len(parent_rate)}")
print(f"names the state overrides             : {len(state_rate & parent_rate)}")
print(f"names only the state has              : {len(state_rate - parent_rate)}")
print(f"names it inherits untouched           : {len(parent_rate - state_rate)}")

### `parent_table()` exists so you can see what was overridden

When a state overrides a table, the countrywide original is still there. Being able to fetch both is how you tell a deliberate deviation from a mistake.

In [ ]:
overridden = sorted(state_rate & parent_rate)
print(f"{len(overridden)} overridden rate tables; first few: {overridden[:5]}\n")

if overridden:
    n = overridden[0]
    mine   = book.table(n, "Rate")
    theirs = book.parent_table(n, "Rate")
    print(f"{n}")
    print(f"   state      : {len(mine.rows):>5} rows   from {mine.package}")
    print(f"   countrywide: {len(theirs.rows):>5} rows   from {theirs.package}")

### Provenance for any value

`cite()` builds the `Citation` that [`03-domain-cell`](03-domain-cell.ipynb) requires. This is the join between *the content* and *a value that can name its source*.

In [ ]:
cite = book.cite("DedFactorProdsCSL", "Rate", locator="row 12")
print(cite)

## What it refuses

Asking for a table neither layer has is an error, not an empty result — and the message names both layers it looked in.

In [ ]:
from gl_engine.errors import TableError

try:
    book.table("NoSuchTableAnywhere", "Rate")
except TableError as e:
    print("TableError:\n ", e)

## Try it yourself

1. Which jurisdiction overrides the most countrywide rate tables? Which overrides the fewest?
2. Find a state override that is **empty** — zero rows. What does that mean for the coverage it prices?
3. Compare `declares()` across two states for the same table name. Do they agree?

In [ ]:
# your turn